In [ ]:
# for debugging
# import sys, platform
# print(sys.executable)
# print(platform.architecture())


In [ ]:
# PyTorch is our core deep learning library
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

print(torch.__version__)


In [ ]:
# Transforms convert the raw image pixels into PyTorch tensors normalized between 0 and 1.
transform = transforms.ToTensor()

# We use the classic MNIST dataset of handwritten digits.
trainset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)

# DataLoaders help us iterate through the dataset in batches (e.g., 128 images at a time) during training.
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)


In [ ]:
# Variational Autoencoder Architecture
# A VAE learns a compressed 'latent' representation of data, which can then be sampled to generate new data.
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super(VAE, self).__init__()
        
        # The Encoder shrinks the 28x28 image down into a latent space representation.
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 400),
            nn.ReLU(),
            # Output is size 40: the first 20 values represent the Mean (mu), 
            # and the last 20 represent the Log Variance (log_var) of the latent distribution.
            nn.Linear(400, 40)  
        )
        
        # The Decoder expands a sampled latent vector back into a 28x28 image.
        self.decoder = nn.Sequential(
            nn.Linear(20, 400),
            nn.ReLU(),
            nn.Linear(400, 28 * 28),
            nn.Sigmoid()  # Sigmoid ensures output pixel values are between 0 and 1.
        )

    # The Reparameterization Trick allows us to backpropagate gradients through a random sampling process.
    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std) # Sample from a standard normal distribution
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, log_var = h[:, :20], h[:, 20:]
        z = self.reparameterize(mu, log_var)
        return self.decoder(z), mu, log_var

# Initialize the model and move it to the GPU if available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VAE().to(device)

# We use the Adam optimizer to iteratively update the model's weights.
optimizer = optim.Adam(vae.parameters(), lr=0.001)

# The VAE Loss function combines two parts:
# 1. Reconstruction Loss (BCE): How accurately did we recreate the original image?
# 2. Kullback-Leibler (KL) Divergence: How closely does our latent space resemble a normal distribution?
def vae_loss(recon_x, x, mu, log_var):
    bce_loss = nn.functional.binary_cross_entropy(recon_x, (x + 1) / 2, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return bce_loss + kl_loss



In [ ]:
# The Training Loop
num_epochs = 10  # An epoch is one full pass through the dataset
for epoch in range(num_epochs):
    for images, _ in trainloader:
        images = images.view(images.size(0), -1).to(device)
        
        # 1. Clear previous gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: generate images and distributions
        recon_images, mu, log_var = vae(images.to(device))
        
        # 3. Calculate loss
        loss = vae_loss(recon_images, images, mu, log_var)
        
        # 4. Backward pass: compute gradients
        loss.backward()
        
        # 5. Optimizer step: update weights
        optimizer.step()
        
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


In [ ]:
# Set the model to evaluation mode (turns off dropout/batchnorm training features)
vae.eval()

# To generate completely new, fake digits, we sample random noise from the latent space!
z = torch.randn(16, 20).to(device)  

# We don't need gradients for inference
with torch.no_grad():
    # Pass the random noise through the Decoder to generate images
    generated_images = vae.decoder(z).view(-1, 1, 28, 28)
